# Week 10, Day 3: Convolution: smoothing spike trains

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w010/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Convolution from scratch

Write `my_convolve(f, g)` using the 'spread every input' picture: the output has length len(f) + len(g) − 1, and each `f[m]` adds `f[m] * g` into the output starting at position m. Check it against `np.convolve`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def my_convolve(f, g):
    f, g = np.asarray(f, dtype=float), np.asarray(g, dtype=float)
    out = np.zeros(len(f) + len(g) - 1)
    for m in range(len(f)):
        ...
    return out

print(my_convolve([1, 2, 3], [0, 1, 0.5]), np.convolve([1, 2, 3], [0, 1, 0.5]))

In [ ]:
import numpy as np
def test_small():
    "my_convolve matches np.convolve on a small example"
    assert np.allclose(my_convolve([1.0, 2.0, 3.0], [0.0, 1.0, 0.5]), np.convolve([1.0, 2.0, 3.0], [0.0, 1.0, 0.5]))

def test_random():
    "my_convolve matches np.convolve on random sequences of different lengths"
    r = np.random.default_rng(0)
    a, b = r.normal(size=17), r.normal(size=5)
    assert np.allclose(my_convolve(a, b), np.convolve(a, b))

def test_commutes():
    "convolution commutes"
    a, b = np.array([1.0, -2.0, 0.5, 3.0]), np.array([2.0, 1.0])
    assert np.allclose(my_convolve(a, b), my_convolve(b, a))

test_small()
test_random()
test_commutes()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(2010)
dt, n = 0.001, 1000                      # 1 ms bins, 1 s
t = np.arange(n) * dt
true_rate = 10 + 40 * np.exp(-(t - 0.35) ** 2 / (2 * 0.04 ** 2)) + 25 * np.exp(-(t - 0.7) ** 2 / (2 * 0.1 ** 2))
counts = (rng.random((20, n)) < true_rate * dt).astype(float)     # 20 trials of spikes in 1 ms bins
psth_raw = counts.mean(axis=0) / dt                                # the raw PSTH, in Hz
print("mean spikes per trial:", counts.sum(axis=1).mean(), "  expected:", round(true_rate.sum() * dt, 1), " (the mean of 20 Poisson counts varies by about 1)")

### Exercise: Smooth with Gaussians of different widths

Write `gaussian_kernel(w)`, a Gaussian with standard deviation w seconds sampled every dt out to ±4w and normalised to area 1. Smooth `psth_raw` with each width in `widths` (use `np.convolve` with `mode='same'` and multiply the kernel by dt), and record the mean squared error against `true_rate` in `mse`.

In [ ]:
def gaussian_kernel(w):
    s = np.arange(-4 * w, 4 * w + dt / 2, dt)
    k = np.exp(-s ** 2 / (2 * w ** 2))
    return ...

widths = [0.002, 0.005, 0.01, 0.02, 0.04, 0.08]
mse = []
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, true_rate, "k--", lw=1, label="true")
for w in widths:
    est = ...
    mse.append(np.mean((est - true_rate) ** 2))
    if w in (0.002, 0.02, 0.08):
        ax.plot(t, est, lw=1.2, label=f"{int(w * 1000)} ms")
ax.set_xlabel("time (s)"); ax.set_ylabel("rate (Hz)"); ax.legend(frameon=False, ncol=4); plt.show()
print("MSE by width:", dict(zip([int(w * 1000) for w in widths], np.round(mse, 1).tolist())))

In [ ]:
import numpy as np
def test_kernel():
    "gaussian_kernel has area 1 and is symmetric"
    k = gaussian_kernel(0.01)
    assert abs(k.sum() * dt - 1) < 1e-9 and np.allclose(k, k[::-1])

def test_mse():
    "mse holds the error against the true rate for each width"
    assert len(mse) == len(widths)
    est = np.convolve(psth_raw, gaussian_kernel(0.02) * dt, mode="same")
    assert np.isclose(mse[widths.index(0.02)], np.mean((est - true_rate) ** 2))

def test_best():
    "the best width is intermediate, neither the narrowest nor the widest"
    assert 0 < int(np.argmin(mse)) < len(widths) - 1

test_kernel()
test_mse()
test_best()
print("All checks passed")

### Exercise: Cross-validate the kernel width

Using even trials (`counts[::2]`) to build the estimate and odd trials (`counts[1::2]`) as the test, compute the cross-validated error for each width in `cv_err`, and store the width that minimises it in `width_cv`. Compare it with the truly best width from the previous exercise.

In [ ]:
psth_A = counts[::2].mean(axis=0) / dt
psth_B = counts[1::2].mean(axis=0) / dt
cv_err = []
for w in widths:
    ...
width_cv = ...

plt.figure(figsize=(5, 3)); plt.semilogx(np.array(widths) * 1000, cv_err, "o-"); plt.xlabel("kernel width (ms)"); plt.ylabel("cross-validated error"); plt.show()
print("cross-validated choice:", int(width_cv * 1000), "ms;  truly best:", int(widths[int(np.argmin(mse))] * 1000), "ms")

In [ ]:
import numpy as np
def test_cv():
    "cv_err compares the smoothed even-trial PSTH with the raw odd-trial PSTH"
    A, B = counts[::2].mean(axis=0) / dt, counts[1::2].mean(axis=0) / dt
    est = np.convolve(A, gaussian_kernel(0.01) * dt, mode="same")
    assert np.isclose(cv_err[widths.index(0.01)], np.mean((est - B) ** 2))

def test_choice():
    "the cross-validated width is within a factor of 2 of the truly best width"
    best = widths[int(np.argmin(mse))]
    assert best / 2 <= width_cv <= best * 2

test_cv()
test_choice()
print("All checks passed")